In [1]:
import pandas as pd
from hatua_ml.data.load_kdhs import load_kdhs

d = load_kdhs()
d["age_group"] = pd.cut(d["age_months"], [23, 35, 47, 59], labels=["24-35", "36-47", "48-59"])
d["sex_label"] = d["sex"].map({1: "Male", 2: "Female"})
d["residence_label"] = d["residence"].map({1: "Urban", 2: "Rural"})

published = {
    "Total": 77.9, "24-35": 82.8, "36-47": 77.4, "48-59": 73.8,
    "Male": 76.3, "Female": 79.7, "Urban": 87.4, "Rural": 72.7,
}

def weighted_rate(g):
    return (g["on_track"] * g["weight"]).sum() / g["weight"].sum() * 100

rows = [("Total", weighted_rate(d), d["weight"].sum())]
for col in ["age_group", "sex_label", "residence_label"]:
    for category, g in d.groupby(col, observed=True):
        rows.append((category, weighted_rate(g), g["weight"].sum()))

check = pd.DataFrame(rows, columns=["category", "ours_%", "weighted_n"])
check["published_%"] = check["category"].map(published)
check["diff"] = check["ours_%"] - check["published_%"]
check.round(1)

,category,ours_%,weighted_n,published_%,diff
0,Total,77.1,4790.2,77.9,-0.8
1,24-35,82.0,1532.8,82.8,-0.8
2,36-47,76.6,1692.9,77.4,-0.8
3,48-59,72.8,1564.5,73.8,-1.0
4,Female,78.9,2309.9,79.7,-0.8
5,Male,75.4,2480.3,76.3,-0.9
6,Rural,71.6,3077.2,72.7,-1.1
7,Urban,87.0,1713.0,87.4,-0.4


In [2]:
from hatua_ml.features.ecdi2030 import ITEMS, cutoff_for_age

items = d[ITEMS].astype(float)
cutoff = cutoff_for_age(d["age_months"])
w = d["weight"]

def weighted_pct(flag):
    return (flag.astype(float) * w).sum() / w.sum() * 100

has_dk = items.isna().any(axis=1)
print(f"children with at least one don't-know: {has_dk.sum()} ({has_dk.mean():.1%})")

variants = {
    "don't-know = not achieved (current)": items.fillna(0).sum(axis=1) >= cutoff,
    "don't-know = achieved": items.fillna(1).sum(axis=1) >= cutoff,
}
for name, flag in variants.items():
    print(f"{name}: {weighted_pct(flag):.1f}%   (published 77.9%)")
    

children with at least one don't-know: 283 (5.2%)
don't-know = not achieved (current): 77.1%   (published 77.9%)
don't-know = achieved: 77.4%   (published 77.9%)
